# 08 Continuous-Time Macroeconomics: HJB and Fokker-Planck Methods

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/04-Macro-Models/08_Continuous_Time_Macro_HJB.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=04-Macro-Models/08_Continuous_Time_Macro_HJB.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: Solving Household Behavior When Time Becomes Continuous
A discrete Bellman equation asks what an agent does between dates. Continuous time instead asks what infinitesimal drift is optimal at every state. This change is computationally valuable because the Hamilton-Jacobi-Bellman (HJB) equation becomes a differential equation and the stationary distribution is characterized by its adjoint Kolmogorov Forward (Fokker-Planck) equation. Together they form the workhorse numerical system behind modern continuous-time heterogeneous-agent macroeconomics.

The economic question is not simply whether we can solve a partial differential equation. It is whether the numerical discretization respects the direction of optimal asset drift, the borrowing boundary, probability conservation, and market-clearing objects. A centered difference can look smooth while violating monotonicity; an upwind scheme chooses the derivative implied by the direction of savings or dissavings. We therefore solve a two-income-state consumption-saving problem with a monotone upwind finite-difference method, verify the HJB residual, recover the invariant distribution from the generator's adjoint, and report aggregate assets.

**Economic question.** In *08 Continuous-Time Macroeconomics: HJB and Fokker-Planck Methods*, what must remain economically invariant when the computational representation changes? Macroeconomic models are useful when their equilibrium restrictions can be traced from household and firm decisions to aggregate dynamics. The key question is which mechanism moves consumption, investment, employment, prices, or distributions after a shock or policy change—and which assumptions are responsible for that response. Comparative statics and impulse responses should therefore be read as disciplined counterfactuals, not as decorative plots.

### Learning Objectives
- **Derive** the stationary continuous-time HJB equation from the dynamic programming principle.
- **Implement** an upwind finite-difference discretization with explicit borrowing and upper-grid boundaries.
- **Recover** the stationary density from the transpose of the Markov generator.
- **Diagnose** a solution using HJB residuals, mass conservation, and boundary drift checks.

### Prerequisites
- `06_Heterogeneous_Agent_Models.ipynb`: incomplete-markets household problem and stationary equilibrium.
- `../03-Economic-Modeling/01_Dynamic_Programming.ipynb`: Bellman equations and contraction logic.
- `../02-Numerical-Methods/08_Differential_Equations.ipynb`: finite differences and stability.
* **Learning-path prerequisite:** [`07_Endogenous_Growth.ipynb`](07_Endogenous_Growth.ipynb)


> **Learning path:** Building on [`07_Endogenous_Growth.ipynb`](07_Endogenous_Growth.ipynb); this notebook closes the current track.


## Table of Contents

1. [Continuous-time Bellman equation](#continuous-time-bellman-equation)
2. [Upwind discretization](#upwind-discretization)
3. [Executable HJB solver](#executable-hjb-solver)
4. [Stationary Fokker-Planck distribution](#stationary-fokker-planck-distribution)
5. [Diagnostics and economic interpretation](#diagnostics-and-economic-interpretation)
6. [Exercises](#exercises)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 12, "figure.figsize": (10, 6), "figure.dpi": 120})
np.set_printoptions(suppress=True, precision=5, linewidth=120)

from scipy.sparse import bmat, diags, eye
from scipy.sparse.linalg import spsolve


<a id="continuous-time-bellman-equation"></a>
## 1. Continuous-Time Bellman Equation

Let assets evolve according to

$$da_t = [y(z_t) + r a_t - c_t]dt,$$

where the income state $z_t\in\{1,2\}$ jumps according to a continuous-time Markov chain with rates $\lambda_{12}$ and $\lambda_{21}$. With CRRA flow utility $u(c)=c^{1-\gamma}/(1-\gamma)$ and discount rate $\rho$, the stationary HJB equation is

$$\rho V_j(a)=\max_{c>0}\left\{u(c)+V_j'(a)[y_j+ra-c]+\lambda_{jk}[V_k(a)-V_j(a)]\right\}.$$

The first-order condition is $u'(c)=V_j'(a)$, hence

$$c_j(a)=\left[V_j'(a)\right]^{-1/\gamma}.$$

The derivative must be chosen consistently with the drift $s_j(a)=y_j+ra-c_j(a)$. If $s>0$, information should arrive from the lower asset node; if $s<0$, it should arrive from the upper node. This is the economic content of the **upwind** rule.


<a id="upwind-discretization"></a>
## 2. Upwind Discretization

On a grid $a_0<\cdots<a_{N-1}$ with spacing $\Delta a$, define forward and backward derivatives

$$D^+V_i=\frac{V_{i+1}-V_i}{\Delta a},\qquad D^-V_i=\frac{V_i-V_{i-1}}{\Delta a}.$$

Candidate consumptions and drifts are $c^+=(D^+V)^{-1/\gamma}$, $s^+=y+ra-c^+$ and analogously for the backward derivative. We choose $D^+V$ where $s^+>0$, $D^-V$ where $s^-<0$, and the steady-state marginal utility $u'(y+ra)$ when neither direction is active. The resulting drift coefficients form a sparse Markov generator $A(V)$.

A false-transient step solves

$$\left[(\rho+\Delta^{-1})I-A(V^n)\right]V^{n+1}=u(c^n)+\Delta^{-1}V^n.$$

This implicit step is much more stable than naively marching the nonlinear HJB forward.


<a id="executable-hjb-solver"></a>
## 3. Executable HJB Solver

The implementation below follows the discrete generator directly. The high-asset boundary is intentionally far from the mass of the stationary distribution; the low-asset boundary enforces the borrowing constraint by replacing the outward derivative with marginal utility from consuming cash-on-hand.


In [ ]:
def solve_two_state_hjb(
    n_assets=240,
    a_min=0.0,
    a_max=30.0,
    income=(0.8, 1.2),
    switch_rates=(0.15, 0.10),
    r=0.03,
    rho=0.05,
    gamma=2.0,
    false_step=1000.0,
    tol=1e-8,
    max_iter=500,
):
    """Solve a stationary two-income-state consumption-saving HJB."""
    if not (0 < r < rho < 1):
        raise ValueError("For this calibration require 0 < r < rho < 1.")
    if gamma <= 0 or n_assets < 20 or a_max <= a_min:
        raise ValueError("Invalid curvature or asset grid.")

    a = np.linspace(a_min, a_max, n_assets)
    da = a[1] - a[0]
    y = np.asarray(income, dtype=float)
    lam12, lam21 = map(float, switch_rates)
    if np.any(y <= 0) or min(lam12, lam21) <= 0:
        raise ValueError("Income and switching rates must be positive.")

    cash = y[:, None] + r * a[None, :]
    utility = lambda c: np.where(gamma == 1.0, np.log(c), c ** (1 - gamma) / (1 - gamma))
    marginal = lambda c: c ** (-gamma)

    # Consuming cash-on-hand forever is a stable initial value guess.
    V = utility(cash) / rho
    generator = None
    consumption = None

    for iteration in range(1, max_iter + 1):
        d_forward = np.empty_like(V)
        d_backward = np.empty_like(V)
        d_forward[:, :-1] = (V[:, 1:] - V[:, :-1]) / da
        d_backward[:, 1:] = (V[:, 1:] - V[:, :-1]) / da
        d_forward[:, -1] = marginal(cash[:, -1])
        d_backward[:, 0] = marginal(cash[:, 0])

        d_forward = np.maximum(d_forward, 1e-12)
        d_backward = np.maximum(d_backward, 1e-12)
        c_forward = d_forward ** (-1.0 / gamma)
        c_backward = d_backward ** (-1.0 / gamma)
        drift_forward = cash - c_forward
        drift_backward = cash - c_backward

        use_forward = drift_forward > 0
        use_backward = drift_backward < 0
        # If both one-sided candidates point inward, prefer the one with larger |drift|.
        conflict = use_forward & use_backward
        choose_forward = use_forward & (~conflict | (np.abs(drift_forward) >= np.abs(drift_backward)))
        choose_backward = use_backward & ~choose_forward
        d_steady = marginal(cash)
        derivative = np.where(choose_forward, d_forward, np.where(choose_backward, d_backward, d_steady))
        consumption = np.maximum(derivative, 1e-12) ** (-1.0 / gamma)
        drift = cash - consumption

        # Prevent probability flow out of the artificial grid.
        drift[:, 0] = np.maximum(drift[:, 0], 0.0)
        drift[:, -1] = np.minimum(drift[:, -1], 0.0)

        blocks = []
        for j, lam in enumerate((lam12, lam21)):
            up = np.maximum(drift[j], 0.0) / da
            down = np.maximum(-drift[j], 0.0) / da
            main = -(up + down + lam)
            blocks.append(diags([down[1:], main, up[:-1]], [-1, 0, 1], shape=(n_assets, n_assets), format="csr"))
        switch12 = lam12 * eye(n_assets, format="csr")
        switch21 = lam21 * eye(n_assets, format="csr")
        generator = bmat([[blocks[0], switch12], [switch21, blocks[1]]], format="csr")

        lhs = (rho + 1.0 / false_step) * eye(2 * n_assets, format="csr") - generator
        rhs = utility(consumption).reshape(-1) + V.reshape(-1) / false_step
        V_new = spsolve(lhs, rhs).reshape(2, n_assets)
        error = float(np.max(np.abs(V_new - V)))
        V = V_new
        if error < tol:
            break
    else:
        raise RuntimeError(f"HJB did not converge after {max_iter} iterations; final error={error:.3e}")

    # Residual using the final generator and consumption.
    residual = rho * V.reshape(-1) - (utility(consumption).reshape(-1) + generator @ V.reshape(-1))
    return {
        "assets": a,
        "value": V,
        "consumption": consumption,
        "generator": generator,
        "iterations": iteration,
        "value_change": error,
        "max_hjb_residual": float(np.max(np.abs(residual))),
    }

hjb = solve_two_state_hjb()
print({k: hjb[k] for k in ("iterations", "value_change", "max_hjb_residual")})
assert hjb["max_hjb_residual"] < 1e-4


In [ ]:
fig, ax = plt.subplots()
for j, label in enumerate(["low income", "high income"]):
    ax.plot(hjb["assets"], hjb["consumption"][j], label=label)
ax.plot(hjb["assets"], 0.8 + 0.03 * hjb["assets"], "--", alpha=0.6, label="low-state cash-on-hand")
ax.set(xlabel="assets", ylabel="consumption", title="Optimal consumption policies from the HJB")
ax.legend()
plt.show()


<a id="stationary-fokker-planck-distribution"></a>
## 4. Stationary Fokker-Planck Distribution

The same generator that moves the value function moves probability mass in the opposite (adjoint) direction. If $g$ stacks probability mass over income and asset states, a stationary distribution satisfies

$$A^\top g=0,\qquad \mathbf{1}^\top g=1.$$

This duality is a powerful correctness check: if the HJB and Kolmogorov equations are discretized inconsistently, the implied distribution often leaks mass or places probability at the artificial grid boundary.


In [ ]:
def stationary_distribution(generator, da):
    """Solve A.T g = 0 with one row replaced by the normalization condition."""
    matrix = generator.T.tolil(copy=True)
    rhs = np.zeros(matrix.shape[0])
    matrix[0, :] = np.ones(matrix.shape[1])
    rhs[0] = 1.0
    mass = spsolve(matrix.tocsr(), rhs)
    mass = np.maximum(mass, 0.0)
    mass /= mass.sum()
    density = mass / da
    return mass, density

da = hjb["assets"][1] - hjb["assets"][0]
mass, density = stationary_distribution(hjb["generator"], da)
n = len(hjb["assets"])
aggregate_assets = float(np.dot(mass[:n] + mass[n:], hjb["assets"]))
boundary_mass = float((mass[0] + mass[n] + mass[n-1] + mass[-1]))
print(f"aggregate assets = {aggregate_assets:.4f}")
print(f"total mass = {mass.sum():.12f}; boundary mass = {boundary_mass:.4%}")
assert np.isclose(mass.sum(), 1.0, atol=1e-10)
assert np.all(mass >= -1e-12)


In [ ]:
fig, ax = plt.subplots()
ax.plot(hjb["assets"], density[:n], label="low income")
ax.plot(hjb["assets"], density[n:], label="high income")
ax.set(xlabel="assets", ylabel="stationary density", title="Invariant asset distribution")
ax.legend()
plt.show()


<a id="diagnostics-and-economic-interpretation"></a>
## 5. Diagnostics and Economic Interpretation

Three checks should accompany every reported equilibrium:

1. **HJB residual:** a small nonlinear residual verifies the discretized optimality equation, not only successive-iteration convergence.
2. **Probability conservation:** stationary mass must be nonnegative and sum to one.
3. **Boundary mass:** substantial probability on the upper grid boundary is evidence that `a_max` is too low and the solution should be recomputed on a wider grid.

A full stationary general equilibrium would add an outer root finder for the interest rate so aggregate household assets equal the economy's asset supply. That extra equilibrium loop is intentionally separated from the household solver here so each numerical layer can be validated independently.


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$da_t = [y(z_t) + r a_t - c_t]dt,$$

**2. Core relation**

$$\rho V_j(a)=\max_{c>0}\left\{u(c)+V_j'(a)[y_j+ra-c]+\lambda_{jk}[V_k(a)-V_j(a)]\right\}.$$

**3. Core relation**

$$c_j(a)=\left[V_j'(a)\right]^{-1/\gamma}.$$

**4. Core relation**

$$D^+V_i=\frac{V_{i+1}-V_i}{\Delta a},\qquad D^-V_i=\frac{V_i-V_{i-1}}{\Delta a}.$$


## Exercises

**1. Upwind logic (Conceptual):** Derive why positive asset drift requires a backward-looking value derivative and negative drift requires a forward-looking derivative. What numerical pathology can a centered derivative create near a kink?

**2. Grid adequacy (Applied):** Re-solve the model with `a_max` equal to 10, 20, 30, and 50. Record aggregate assets, maximum HJB residual, and boundary mass. Identify the smallest grid that produces a stable aggregate statistic.

**3. Market clearing (Challenge):** Wrap `solve_two_state_hjb` in a scalar root finder for `r`. Specify an exogenous asset supply, solve for the clearing rate, and verify that both the household residual and market-clearing residual meet stated tolerances.


## Summary & Key Takeaways

- Continuous-time household optimization is summarized by an HJB equation; distribution dynamics use the adjoint Kolmogorov equation.
- Upwinding is an economic as well as numerical restriction because the derivative must follow the direction of optimal drift.
- False-transient implicit steps produce a stable sparse linear solve inside the nonlinear fixed point.
- HJB residuals, mass conservation, and boundary mass provide independent evidence that the computed solution is usable.


## References & Further Reading

- Achdou, Y., Han, J., Lasry, J.-M., Lions, P.-L. & Moll, B. (2022). Income and wealth distribution in macroeconomics: A continuous-time approach. *Review of Economic Studies*, 89(1), 45–86.
- Moll, B. Continuous-time heterogeneous-agent methods and numerical notes.
- Aiyagari, S. R. (1994). Uninsured idiosyncratic risk and aggregate saving. *Quarterly Journal of Economics*, 109(3), 659–684.
